## Homework 5: Monitoring

This is the personal implementation of the homework 5 of [LLM Zoomcamp 2026 Cohort](https://github.com/DataTalksClub/llm-zoomcamp/tree/main). In this homework, the goal is to add observability to the RAG application. While previous modules focused on building and evaluating the system, this module focuses on understanding how the application behaves while it is running. To achieve this, the RAG pipeline is instrumented using OpenTelemetry (OTel). Instead of changing the RAG logic, tracing is added around the main operations so that each execution can be monitored. It reuses the same data and the same search functions from homework 2. The course repository is organized by modules and each module is a top-level folder with a `lessons/` subfolder of numbered markdown pages:

```python
01-agentic-rag/
└── lessons/
    ├── 01-intro.md
    ├── 02-environment.md
    ├── ...
    └── 16-other-frameworks.md
```

Each lesson page is a single markdown file, which are exactly the data to be fetched from GitHub and to be used as the knowledge base. 

### Q1. First trace

Wrap the rag() method so each call produces a span. The simplest way is to create a RAGTraced subclass of RAGBase that wraps rag(), search(), and llm() each in their own span.

Run this query:

    How does the agentic loop keep calling the model until it stops?

The console exporter prints every finished span as a dictionary. Count the spans in the console output - each one is a separate ReadableSpan entry. How many spans does the trace produce?

In [2]:
%run starter.py

Overriding of current TracerProvider is not allowed


{
    "name": "search",
    "context": {
        "trace_id": "0x5fe954270381f215d92aa356fc0e24d2",
        "span_id": "0xc95c92a7b6d677fd",
        "trace_state": "[]"
    },
    "kind": "SpanKind.INTERNAL",
    "parent_id": "0xf638876630968950",
    "start_time": "2026-07-20T19:26:31.972459Z",
    "end_time": "2026-07-20T19:26:31.980955Z",
    "status": {
        "status_code": "UNSET"
    },
    "attributes": {
        "search.query": "How does the agentic loop keep calling the model until it stops?",
        "search.num_results": 5,
        "search.results_length": 5
    },
    "events": [],
    "links": [],
    "resource": {
        "attributes": {
            "telemetry.sdk.language": "python",
            "telemetry.sdk.name": "opentelemetry",
            "telemetry.sdk.version": "1.44.0",
            "service.instance.id": "c19c56de-35ac-40b6-9ec5-c092e53f1659",
            "service.name": "unknown_service"
        },
        "schema_url": ""
    }
}
{
    "name": "llm",
    "co

The `ConsoleSpanExporter` prints one dictionary for every completed span. Since the `RAGTraced` class instruments the `rag()`, `search()`, and `llm()` methods, executing a single query produces **three spans**.

The trace consists of:

- **`rag`** – The root (parent) span representing the complete execution of the RAG pipeline. It records attributes such as the input query and the length of the generated response.
- **`search`** – A child span representing the document retrieval step. It records the search query, the requested number of search results, and the number of retrieved documents.
- **`llm`** – A child span representing the call to the language model. It records the language model used to generate the response.

The console output contains exactly **three completed spans**:

1. `search`
2. `llm`
3. `rag`

Therefore, the correct answer to the homework question is **3 spans**.

The text printed after the span dictionaries is the final answer generated by the RAG application. It is **not** part of the trace and should not be counted as an additional span.

### Q2. Capturing metrics as span attributes

Read the token usage from the LLM response (the llm() method in the starter already returns the raw response object) and set them as attributes on the llm span. Now re-run the query. How many input tokens do we see?

In [5]:
%run starter.py

Overriding of current TracerProvider is not allowed


{
    "name": "search",
    "context": {
        "trace_id": "0xf6105bc22dff25d7b48be627bf7e4298",
        "span_id": "0x7757925c296e3585",
        "trace_state": "[]"
    },
    "kind": "SpanKind.INTERNAL",
    "parent_id": "0x6bf25aba4d893ec9",
    "start_time": "2026-07-20T19:48:32.332212Z",
    "end_time": "2026-07-20T19:48:32.334284Z",
    "status": {
        "status_code": "UNSET"
    },
    "attributes": {
        "search.query": "How does the agentic loop keep calling the model until it stops?",
        "search.num_results": 5,
        "search.results_length": 5
    },
    "events": [],
    "links": [],
    "resource": {
        "attributes": {
            "telemetry.sdk.language": "python",
            "telemetry.sdk.name": "opentelemetry",
            "telemetry.sdk.version": "1.44.0",
            "service.instance.id": "c19c56de-35ac-40b6-9ec5-c092e53f1659",
            "service.name": "unknown_service"
        },
        "schema_url": ""
    }
}
{
    "name": "llm",
    "co

To enrich the trace with additional monitoring information, the `llm()` span was extended to record token usage and request cost. The token counts were obtained from the OpenAI response object (`response.usage`) and added as span attributes using `set_attribute()`. The request cost was then calculated using the pricing function implemented in a previous module and recorded as additional span attributes.

The `llm` span now includes the following attributes:

- `input_tokens`
- `output_tokens`
- `llm.input_cost`
- `llm.output_cost`
- `llm.total_cost`

After rerunning the query, the trace produced the following values:

```text
input_tokens: 7111
output_tokens: 113
llm.input_cost: 0.00533325
llm.output_cost: 0.0005085
llm.total_cost: 0.00584175
```

The homework asks for the closest option rather than the exact number of tokens. Since the recorded input token count is **7111**, the closest answer is **7000**.

### Q3. Span timing

Each span automatically records its duration. Look at the console output from Q1 and find the durations for the search span and the llm span.

For a typical query, roughly how long does the LLM call take?

In [6]:
%run starter.py

Overriding of current TracerProvider is not allowed


{
    "name": "search",
    "context": {
        "trace_id": "0xd6013c1f97182b347ed6d86121f65a1c",
        "span_id": "0x48b99b65e01e2c72",
        "trace_state": "[]"
    },
    "kind": "SpanKind.INTERNAL",
    "parent_id": "0xe6245abf14992eae",
    "start_time": "2026-07-20T19:52:33.338372Z",
    "end_time": "2026-07-20T19:52:33.340460Z",
    "status": {
        "status_code": "UNSET"
    },
    "attributes": {
        "search.query": "How does the agentic loop keep calling the model until it stops?",
        "search.num_results": 5,
        "search.results_length": 5
    },
    "events": [],
    "links": [],
    "resource": {
        "attributes": {
            "telemetry.sdk.language": "python",
            "telemetry.sdk.name": "opentelemetry",
            "telemetry.sdk.version": "1.44.0",
            "service.instance.id": "c19c56de-35ac-40b6-9ec5-c092e53f1659",
            "service.name": "unknown_service"
        },
        "schema_url": ""
    }
}
{
    "name": "llm",
    "co

OpenTelemetry automatically records the start and end timestamps for every span, allowing the execution time of each operation to be measured without adding any additional code.

The recorded timestamps for the `llm` span were:

```text
start_time: 2026-07-20T19:52:33.342328Z
end_time:   2026-07-20T19:52:35.524495Z
```

The duration is therefore:

```text
35.524495 - 33.342328 ≈ 2.182 seconds
≈ 2182 ms
```

The `search` span completed much faster:

```text
start_time: 2026-07-20T19:52:33.338372Z
end_time:   2026-07-20T19:52:33.340460Z
Duration ≈ 2.1 ms
```

After running the query multiple times, the `llm` span consistently took just over **2 seconds** to complete. Therefore, the correct answer is:

**✅ Over 2000 ms**

### Q4. Saving traces to SQLite

Re-run the query from Q1. Which span names appear in the spans table?

In [ ]:
# no console output expected as spans, but should be written in db file
%run starter.py

The loop keeps calling the model by using a `while True` loop and a `has_function_calls` flag.

- Each iteration sends the full `messages` history to the model.
- If the model returns a `function_call`, the code runs the tool, appends the tool output to `messages`, and sets `has_function_calls = True`.
- If the model returns only a normal `message`, then `has_function_calls` stays `False`.
- At the end of the iteration, if `has_function_calls == False`, the loop `break`s.

So it stops when the model returns a final answer with no more tool calls.


In [2]:
import pandas as pd
import sqlite3

conn = sqlite3.connect("traces.db")

pd.read_sql("SELECT * FROM spans", conn)

,name,start_time,end_time,input_tokens,output_tokens,cost
0,search,1784577957118886823,1784577957125874293,NaN,NaN,NaN
1,llm,1784577957133355653,1784577960321565952,7111.0,134.0,0.005936
2,rag,1784577957118814124,1784577960324033877,NaN,NaN,NaN


Instead of exporting spans to the console, a custom `SQLiteSpanExporter` was implemented by extending OpenTelemetry's `SpanExporter`. The exporter creates a SQLite database (`traces.db`) and stores information about every completed span in a `spans` table.

The exporter records the following information for each span:

- `name`
- `start_time`
- `end_time`
- `input_tokens`
- `output_tokens`
- `cost`

After replacing the `ConsoleSpanExporter` with the `SQLiteSpanExporter` and rerunning the query, the contents of the `spans` table were:

| name | input_tokens | output_tokens | cost |
|------|-------------:|--------------:|-----:|
| search | — | — | — |
| llm | 7111 | 134 | 0.005936 |
| rag | — | — | — |

Only the `llm` span contains token usage and cost because these attributes were explicitly added during the LLM call. The `search` and `rag` spans do not record these metrics, so their corresponding columns contain `NULL` values.

The span names stored in the SQLite database are:

- `search`
- `llm`
- `rag`

Therefore, the correct answer is:

**✅ rag, search, and llm**

### Q5. Querying trace data

The rag span wraps everything, so its duration includes both search and llm. To see where time actually goes, exclude the rag span and compare the children. Using SQL (or pandas), compute the total duration for each span name excluding rag. Which span type takes the most total time?

In [3]:
import pandas as pd
import sqlite3

conn = sqlite3.connect("traces.db")

df = pd.read_sql("SELECT * FROM spans", conn)
df.head()

,name,start_time,end_time,input_tokens,output_tokens,cost
0,search,1784577957118886823,1784577957125874293,NaN,NaN,NaN
1,llm,1784577957133355653,1784577960321565952,7111.0,134.0,0.005936
2,rag,1784577957118814124,1784577960324033877,NaN,NaN,NaN


In [4]:
df["duration_ms"] = (df["end_time"] - df["start_time"]) / 1_000_000

(
    df[df["name"] != "rag"]
    .groupby("name")["duration_ms"]
    .sum()
)

name
llm       3188.210299
search       6.987470
Name: duration_ms, dtype: float64

To determine which operation consumes the most execution time, the spans stored in the SQLite database were analyzed. Since the `rag` span represents the entire pipeline and includes both the search and LLM operations, it was excluded from the analysis.

First, the duration of each span was computed from the recorded timestamps:

```python
df["duration_ms"] = (df["end_time"] - df["start_time"]) / 1_000_000
```

Next, the durations were grouped by span name and summed across all recorded traces:

```python
(
    df[df["name"] != "rag"]
    .groupby("name")["duration_ms"]
    .sum()
)
```

The results were:

| Span | Total Duration (ms) |
|------|--------------------:|
| `llm` | 3188.21 |
| `search` | 6.99 |

The results show that the **LLM call** accounts for nearly all of the execution time, while the **search** operation completes in only a few milliseconds. This indicates that the language model is the primary performance bottleneck in the RAG pipeline.

Therefore, the correct answer is:

**✅ llm**

### Question 6: Token stability across runs

Run the same query from Q1 three more times (so you have 4 RAG calls total in the database). Then compute the input tokens for each llm span. How much do the input tokens vary across these 4 runs?

In [8]:
# run 3 times
%run starter.py

Overriding of current TracerProvider is not allowed


It keeps calling the model in a `while True` loop.

Each iteration:
1. Sends the full message history to the model.
2. Checks the response for any `function_call` items.
3. Runs those tools and appends the tool outputs to memory.
4. If there were function calls, it loops again.
5. If there were no function calls, it `break`s.

So the stop condition is simple: **no function calls in the model response means the agent is done**.


In [9]:
import sqlite3
import pandas as pd

conn = sqlite3.connect("traces.db")

df = pd.read_sql("SELECT * FROM spans", conn)
df

,name,start_time,end_time,input_tokens,output_tokens,cost
0,search,1784577957118886823,1784577957125874293,NaN,NaN,NaN
1,llm,1784577957133355653,1784577960321565952,7111.0,134.0,0.005936
2,rag,1784577957118814124,1784577960324033877,NaN,NaN,NaN
3,search,1784578489618520640,1784578489620988542,NaN,NaN,NaN
4,llm,1784578489624516984,1784578492549746607,7111.0,87.0,0.005725
5,rag,1784578489618438649,1784578492552363772,NaN,NaN,NaN
6,search,1784578536207113929,1784578536208604912,NaN,NaN,NaN
7,llm,1784578536211247521,1784578539021923377,7111.0,100.0,0.005783
8,rag,1784578536207063746,1784578539025520720,NaN,NaN,NaN
9,search,1784578541703120654,1784578541704603617,NaN,NaN,NaN


In [10]:
df[df["name"] == "llm"][["input_tokens", "output_tokens", "cost"]]

,input_tokens,output_tokens,cost
1,7111.0,134.0,0.005936
4,7111.0,87.0,0.005725
7,7111.0,100.0,0.005783
10,7111.0,106.0,0.005810


To evaluate the consistency of the RAG pipeline, the same query was executed four times. The `input_tokens` attribute was retrieved from each `llm` span stored in the SQLite database.

Using pandas:

```python
df[df["name"] == "llm"][["input_tokens", "output_tokens", "cost"]]
```

The recorded values were:

| Run | Input Tokens | Output Tokens | Cost |
|----:|-------------:|--------------:|------:|
| 1 | 7111 | 134 | 0.005936 |
| 2 | 7111 | 87 | 0.005725 |
| 3 | 7111 | 100 | 0.005783 |
| 4 | 7111 | 106 | 0.005810 |

The **input token count remained identical** across all four executions, indicating that the RAG pipeline consistently retrieved the same context for the same query. Although the **output tokens** varied slightly, this is expected because the language model may generate responses of different lengths.

Therefore, the correct answer is:

**✅ They're identical**